# 🛰️🔆 TKG Solar — Eval-only (T4 hoặc CPU, KHÔNG tốn CU)

Notebook **đánh giá riêng**, không train. Nạp checkpoint `best_tkg.pt` đã lưu trên Drive → chạy forward trên **val + test** → ra MAE/RMSE/MAPE theo horizon + bảng benchmark.

**Chạy được trên T4 (free tier) hoặc CPU runtime** — chỉ forward pass, không backward.

Thứ tự: Runtime → (T4 GPU hoặc None/CPU) → Run all.

> ⚠️ **Caveat so sánh:** baselines (`base_results.json`) được eval trên `splits_base` (toàn span, no-sat, scaler riêng); Proposed eval trên `splits_full` (subset khớp Himawari, scaler riêng). Bảng chỉ mang tính **tham khảo** — fair thực sự cần train lại baselines trên `splits_full`.

In [ ]:
!nvidia-smi || echo 'CPU runtime (không có GPU) — vẫn chạy eval được'

In [ ]:
import os
if not os.path.isdir('tkg-solar-satellite-reasoning'):
    !git clone https://github.com/DucLong06/tkg-solar-satellite-reasoning.git
%cd tkg-solar-satellite-reasoning
!git checkout init-project && git pull

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
!pip install -q timm torch_geometric h5py scikit-learn pyyaml tqdm statsmodels

In [ ]:
import pathlib
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = '/content/drive/MyDrive/tkg-solar-data'   # <-- ĐIỀN đúng thư mục Drive của bạn
DKASC_CSV    = f'{DATA_ROOT}/dkasc/alice_2020_2022_clean.csv'
HIMAWARI_DIR = f'{DATA_ROOT}/himawari_alice'
print(('OK  ' if pathlib.Path(DKASC_CSV).exists() else 'MISSING '), DKASC_CSV)
print(('OK  ' if pathlib.Path(f'{HIMAWARI_DIR}/frames.h5').exists() else 'MISSING '), f'{HIMAWARI_DIR}/frames.h5')
print(('OK  ' if pathlib.Path(f'{DATA_ROOT}/checkpoints/best_tkg.pt').exists() else 'MISSING '), f'{DATA_ROOT}/checkpoints/best_tkg.pt')

In [ ]:
from src.training.config import Config
from src.training.gpu_autoscale import resolve_runtime
from src.common.seeding import seed_everything
from src.common.shapes import HORIZON_MINUTES, N_HORIZONS

cfg = Config.from_yaml('configs/paper_config.yaml')
cfg.device = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg.dkasc_csv = DKASC_CSV
cfg.himawari_dir = HIMAWARI_DIR
cfg.checkpoint_dir = f'{DATA_ROOT}/checkpoints'
cfg.cache_dir = 'data/cache'
cfg.pretrained_backbone = False   # eval-only: trọng số ViT đã nằm trong best_tkg.pt -> khỏi tải HF
cfg.auto_batch = False            # eval: batch cố định nhỏ cho T4/CPU
cfg.batch_size = 16
resolve_runtime(cfg)
seed_everything(cfg.seed)
print('device', cfg.device, '| precision', cfg.precision, '| batch', cfg.batch_size)

## 1 · Nạp pipeline (PV + meteo + vệ tinh) — GIỐNG HỆT lúc train

Cùng `k`, ngày split, night-filter, `min_steps`, `img_size=64` → cùng scaler + cùng test windows với lúc train (để val eval khớp ~13.46).

In [ ]:
import time
from src.data_pipeline import DataPipeline
assert pathlib.Path(f'{HIMAWARI_DIR}/frames.h5').exists(), 'Thiếu frames.h5 -> upload Himawari'
print('Loading splits_full (PV+meteo+satellite)...', flush=True); _t = time.time()
splits_full = DataPipeline.load(
    cfg.dkasc_csv, cfg.himawari_dir,
    k=cfg.k, batch_size=cfg.batch_size, img_size=64,
    min_steps=cfg.min_steps, train_end=cfg.train_end, val_end=cfg.val_end,
    cadence_min=cfg.cadence_min, night_ghi_thresh=cfg.night_ghi_thresh,
    cache_dir=cfg.cache_dir, use_satellite=True, scaler_out=cfg.scaler_out,
)
print(f'done {time.time()-_t:.0f}s |', splits_full.meta)

## 2 · Nạp checkpoint + eval VAL & TEST + kiểm tra collapse

- **VAL** phải ra ≈ 13.46 → xác nhận checkpoint + scaler đúng.
- **TEST** = con số để viết báo.
- `ratio = test/val`: <1.5 OK; ≥1.5 nghi ngờ bug pipeline (collapse như journal cũ).

In [ ]:
import json
from pathlib import Path
from src.fusion_predictor.tkg_solar_model import TKGSolarModel
from src.training.train_loop import predict_loader
from src.metrics.regression_metrics import compute_per_horizon

model = TKGSolarModel.from_config(cfg)
ckpt = Path(cfg.checkpoint_dir) / 'best_tkg.pt'
assert ckpt.exists(), f'Thiếu {ckpt} -> copy best_tkg.pt vào {cfg.checkpoint_dir}'
state = torch.load(ckpt, map_location=cfg.device, weights_only=True)
model.load_state_dict(state['model_state']); model.to(cfg.device); model.eval()
print('loaded best_tkg.pt | epoch', state.get('epoch'), '| saved val_mae', round(float(state.get('val_mae', float('nan'))), 4))

def eval_split(loader, name):
    n = len(loader.dataset)
    print(f"\n>>> eval {name}: {n} windows, {len(loader)} batches (CPU chậm, có thanh tqdm bên dưới)", flush=True)
    yt, yp = predict_loader(model, loader, cfg.device, progress=True)  # tqdm bar over batches
    yt = splits_full.scalers.inverse_pv(yt.numpy()); yp = splits_full.scalers.inverse_pv(yp.numpy())
    per = compute_per_horizon(yt, yp, HORIZON_MINUTES, cfg.mape_min_value)
    print(f"[{name}] per-horizon (kW)")
    print(f"{'horizon':>10} | {'MAE':>9} | {'RMSE':>9} | {'MAPE %':>9}")
    print('-' * 46)
    for lab in ['overall', *[f'{m}min' for m in HORIZON_MINUTES]]:
        mm = per[lab]
        print(f"{lab:>10} | {mm['mae']:9.4f} | {mm['rmse']:9.4f} | {mm['mape']:9.2f}")
    return per

val_per  = eval_split(splits_full.val_loader,  'VAL')   # sanity: ~13.46 -> checkpoint+scaler OK
test_per = eval_split(splits_full.test_loader, 'TEST')

vmae, tmae = val_per['overall']['mae'], test_per['overall']['mae']
ratio = tmae / max(vmae, 1e-9)
print(f"\nval_mae={vmae:.3f}  test_mae={tmae:.3f}  ratio={ratio:.2f}")
print('OK: test ≈ val, không collapse' if ratio < 1.5 else 'NGHI NGỜ: test ≫ val -> bug pipeline (collapse như journal cũ), KHÔNG phải kết quả model')

pathlib.Path(f'{DATA_ROOT}/outputs').mkdir(parents=True, exist_ok=True)
json.dump(test_per['overall'], open(f'{DATA_ROOT}/outputs/proposed_result.json', 'w'), indent=2)
print('saved -> proposed_result.json')

## 3 · Bảng benchmark (baselines + Proposed) — tham khảo

In [ ]:
import json
from src.evaluation.benchmark_table import build_benchmark, relative_ordering

out = f'{DATA_ROOT}/outputs'
all_results = {}
if pathlib.Path(f'{out}/base_results.json').exists():
    all_results.update(json.load(open(f'{out}/base_results.json')))
if pathlib.Path(f'{out}/proposed_result.json').exists():
    all_results['Proposed'] = json.load(open(f'{out}/proposed_result.json'))
assert all_results, 'Chưa có kết quả -> chạy cell trên trước'

print(build_benchmark(all_results, scalers=splits_full.scalers))
print('\nThứ tự (tốt→kém MAE):', ' < '.join(relative_ordering(all_results)))
print('\n[CAVEAT] Baselines eval trên splits_base (full span, no-sat, scaler riêng);'
      ' Proposed eval trên splits_full (subset khớp Himawari, scaler riêng).'
      ' So sánh chỉ tham khảo — fair cần train lại baselines trên splits_full.')